# 带有内存作用域基础内存分配器

测试带有内存作用域的基础内存分配器（Naive allocator）在 Relax VM 中的使用。

本文件包含的测试用例，用于验证 Relax 虚拟机中使用基础内存分配器和存储作用域的功能。
主要测试内容：
1. 定义 IR 模块，包含 TIR 原语函数和 Relax 函数
2. 测试带有全局存储作用域（`storage_scope="global"`）的内存分配功能
3. 验证 Relax VM 使用基础内存分配器的正确性

In [1]:
import numpy as np  # 用于生成随机数据和数组操作
import tvm
import tvm.testing  # 包含TVM的测试工具
from tvm import relax  # 导入Relax框架
from tvm.script import ir as I  # 导入IR脚本支持
from tvm.script import relax as R  # 导入Relax脚本支持
from tvm.script import tir as T  # 导入TIR脚本支持

In [2]:
@I.ir_module
class Module:
    @T.prim_func
    def add(
        arg0: T.Buffer((2, 2), "float32"),
        arg1: T.Buffer((2, 2), "float32"),
        output: T.Buffer((2, 2), "float32"),
    ):
        """
        TIR原始函数：执行二维数组的元素级加法
        
        参数:
            arg0: 输入缓冲区1，形状为(2, 2)的float32类型
            arg1: 输入缓冲区2，形状为(2, 2)的float32类型
            output: 输出缓冲区，形状为(2, 2)的float32类型
        """
        # 标记此函数为relax.add操作符的实现
        T.func_attr({"operator_name": "relax.add"})
        
        # 双重循环遍历二维数组的每个元素
        for ax0 in range(2):
            for ax1 in range(2):
                # 定义计算块，指定空间轴和读写依赖
                with T.block("T_add"):
                    v_ax0 = T.axis.spatial(2, ax0)
                    v_ax1 = T.axis.spatial(2, ax1)
                    T.reads(arg0[v_ax0, v_ax1], arg1[v_ax0, v_ax1])
                    T.writes(output[v_ax0, v_ax1])
                    # 执行加法操作
                    output[v_ax0, v_ax1] = arg0[v_ax0, v_ax1] + arg1[v_ax0, v_ax1]

    @R.function(pure=False)
    def main(x: R.Tensor((2, 2), dtype="float32")):
        cls = Module
        # 分配存储空间，大小为2*2=4个float32元素，指定运行时设备索引为0，存储作用域为"global"
        storage = R.vm.alloc_storage(
            R.shape([2 * 2]), runtime_device_index=0, dtype="float32", storage_scope="global"
        )
        # 从存储中分配张量，偏移量为0，形状为[2, 2]，数据类型为float32
        alloc = R.vm.alloc_tensor(storage, offset=0, shape=R.shape([2, 2]), dtype="float32")
        # 调用add函数执行加法操作，将结果存储到alloc中
        _: R.Tuple = cls.add(x, x, alloc)
        # 将alloc作为输出张量
        out: R.Tensor((2, 2), dtype="float32") = alloc
        return out

## 测试带有全局存储作用域的存储分配功能

此测试验证 Relax VM 在使用基础内存分配器时是否正确处理带有指定存储作用域的内存分配，以及后续的计算和数据获取是否正确。

In [3]:
# 生成随机测试数据，形状为(2, 2)的float32类型
arg0 = np.random.uniform(size=(2, 2)).astype(np.float32)
# 计算期望的输出结果（输入自身相加）
output_ref = arg0 + arg0

# 获取IR模块
mod = Module
# 设置目标编译平台为LLVM
target = "llvm"

# 使用优化级别3编译模块
with tvm.transform.PassContext(opt_level=3):
    lib = tvm.relax.build(mod, target=target, exec_mode="compiled")

# 获取CPU设备
dev = tvm.cpu()
# 这是关键行：使用朴素内存分配器创建Relax虚拟机运行时
vm_rt = relax.VirtualMachine(lib, dev, memory_cfg="naive")

# 将输入数据转换为TVM NDArray
x = tvm.nd.array(arg0, dev)
# 设置虚拟机的输入参数
vm_rt.set_input("main", x)
# 调用有状态的main函数
vm_rt.invoke_stateful("main")
# 获取输出结果并转换为numpy数组
output = vm_rt.get_outputs("main").numpy()
# 验证输出结果与期望值是否接近
tvm.testing.assert_allclose(output_ref, output)

InternalError: LLVM module verification failed with the following errors: 
Called function is not the same type as the call!
  %253 = call i32 @add(i8* null, %0* %250, i32 3, %0* %252), !dbg !64
